# Lecture 19

# Exploratory Data Analysis

## Week 7 Friday

## Miles Chen, PhD

We will work through three questions using the same routine:
check what the variables mean, choose relevant records, inspect counts and
summaries, then plot and explain what the result supports.


# Import NHANES Data

We will explore **NHANES 2015-2016**, the National Health and Nutrition
Examination Survey, using two supplied tables:

1. `DEMO_I`: demographics, including recorded sex, age, and household income.
2. `BMX_I`: body measurements, including weight and lengths.

The data dictionaries tell us what the variables and numeric codes mean:

- [Demographic variables](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/DEMO_I.htm)
- [Body measurements](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/BMX_I.htm)

Run this notebook from `live_demos`; the two `.XPT` files are in `../data`.
We will use simple **unweighted sample summaries**. NHANES population estimates
require appropriate survey weights and attention to its sampling design;
see [CDC's weighting tutorial](https://wwwn.cdc.gov/nchs/nhanes/tutorials/weighting.aspx).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# XPT is a SAS transport format, not a CSV.
bmx = pd.read_sas("../data/BMX_I.XPT", format="xport")


In [ ]:
demo = pd.read_sas("../data/DEMO_I.XPT", format="xport")


In [ ]:
# SEQN identifies a participant. Require one record per participant in each table.
print("Body-measurement rows:", len(bmx))
print("Demographic rows:", len(demo))
merged = bmx.merge(demo, on="SEQN", how="inner", validate="one_to_one")
print("Matched participants:", len(merged))


The inner join keeps participants represented in both files. In these data,
all body-measurement records find a demographic match. `validate="one_to_one"`
checks key uniqueness; it does not check that every participant appears in both files.


In [ ]:
# One row now contains a participant's measurements and demographics.
merged.head()


In [ ]:
merged.info()

## Do baby boys weigh more than baby girls on average?

The variables we need are:

| Variable | Meaning |
| --- | --- |
| `RIDAGEMN` | Age in months at screening, released for young children |
| `RIAGENDR` | Recorded male/female category: 1 = male, 2 = female |
| `BMXWT` | Measured body weight, in kilograms |

See the [demographic codebook](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/DEMO_I.htm#RIDAGEMN)
and [weight documentation](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/BMX_I.htm#BMXWT).
We retain screening age for this example; the examination may happen later.
For age at the weight measurement, `RIDEXAGM` would be the relevant age variable.


## Exploration with Seaborn

We first select babies aged 12 months or younger at screening. Seaborn's
`lineplot` averages repeated y-values within each month/sex group.
We will show the same means in pandas and check the group counts.


In [ ]:
# Keep the rows and columns needed for this question; preserve merged.
babies = merged.loc[
    merged["RIDAGEMN"] <= 12, ["RIDAGEMN", "BMXWT", "RIAGENDR"]
].copy()

babies = babies.rename(columns={
    "RIDAGEMN": "age_months",
    "BMXWT": "weight_kg",
    "RIAGENDR": "sex",
})
print("Selected babies:", len(babies))
print("Missing weights:", babies["weight_kg"].isna().sum())


In [ ]:
# These are category codes, so map them to readable labels rather than average them.
babies["sex"] = babies["sex"].map({1: "Boy", 2: "Girl"})


In [ ]:
babies.loc[:, ["age_months", "weight_kg", "sex"]].head()


In [ ]:
# The age filter excludes missing ages; the recorded month values are whole months.
babies["age_months"] = babies["age_months"].astype(int)
babies.loc[:, ["age_months", "weight_kg", "sex"]].head()


In [ ]:
fig, ax = plt.subplots()
sns.lineplot(
    data=babies, x="age_months", y="weight_kg", hue="sex",
    hue_order=["Boy", "Girl"], estimator="mean", errorbar=None, marker="o", ax=ax,
)
ax.set(
    title="Mean recorded weight by screening age and sex",
    xlabel="Age at screening (months)", ylabel="Mean weight (kg)",
)
plt.show()


In [ ]:
# These counts describe the selected records, before any averaging.
babies["sex"].value_counts(dropna=False)


In [ ]:
# size counts rows; count counts recorded weights contributing to each mean.
baby_summary = babies.groupby(["age_months", "sex"], as_index=False).agg(
    n_babies=("weight_kg", "size"),
    n_weights=("weight_kg", "count"),
    mean_weight_kg=("weight_kg", "mean"),
)
baby_summary


In [ ]:
# unstack moves the sex labels into columns: one mean per month/sex pair.
summary_table = babies.groupby(["age_months", "sex"])["weight_kg"].mean().unstack("sex")
summary_table


In [ ]:
# pandas plots the means already computed; it does not recompute them.
ax = summary_table.plot(marker="o")
ax.set(
    title="The same group means, plotted with pandas",
    xlabel="Age at screening (months)", ylabel="Mean weight (kg)",
)
plt.show()


### Comments

In these sampled babies, boys have a higher observed mean weight at every
screening month from 0 through 12. Group sizes vary, so inspect the counts
alongside the means; this does not mean every boy is heavier than every girl.

Each plotted point is a group mean, and the connecting line joins different
babies at different ages. It is not an individual growth trajectory.
`errorbar=None` keeps both plots focused on the descriptive means.


## What is the relationship between height, leg length, and arm length? Does this relationship change with age?

`RIDAGEYR` records age in years at screening. The lengths are in centimeters:
`BMXHT` is standing height, `BMXLEG` is **upper** leg length, and `BMXARML` is
**upper** arm length.

- [Age documentation](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/DEMO_I.htm#RIDAGEYR)
- [Measurement documentation](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/BMX_I.htm)

These are not whole-leg and whole-arm measurements.


We choose adults aged 20 and older and require all three lengths so that
each comparison uses the same records. Dropping rows is a decision about this
question, rather than a rule for every NHANES analysis.


In [ ]:
# Select relevant fields before handling missing values.
limbs = merged.loc[:, ["BMXHT", "BMXLEG", "BMXARML", "RIDAGEYR"]].copy()
limbs = limbs.dropna(subset=["BMXHT", "BMXLEG", "BMXARML", "RIDAGEYR"])
adults = limbs.loc[limbs["RIDAGEYR"] >= 20].copy()
print("Adults with all three lengths:", len(adults))
adults.info()  # info already prints its report; its return value is None.


There are **5,086 adults** with all three lengths recorded in these files.
A question about height alone could retain additional records.


Let's look at the correlation between:

   - standing height and upper leg length
   - standing height and upper arm length
   - upper leg length and upper arm length

In [ ]:
# Rename columns
adults = adults.rename(columns={
    'BMXHT': 'height', 
    'BMXLEG': 'leg_length',
    'BMXARML': 'arm_length',
    'RIDAGEYR': 'age_years'
})

In [ ]:
# Correlate individual measurements, excluding age and numeric category codes.
adults[["height", "leg_length", "arm_length"]].corr()


Use `pd.cut` to create decade groups: 20-29, 30-39, ..., 70-79, and 80+.
With `right=False`, the left endpoint is included and the right endpoint is excluded.

NHANES records all ages 80 and older as 80, so the last label is **80+**.
Explicit labels will stay attached to the groups when we summarize and plot.

[pandas.cut documentation](https://pandas.pydata.org/docs/reference/api/pandas.cut.html)


In [ ]:
age_labels = ["20-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]
agegroups = pd.cut(
    adults["age_years"], bins=[20, 30, 40, 50, 60, 70, 80, np.inf],
    labels=age_labels, right=False,
)


In [ ]:
agegroups.head()


In [ ]:
# The category Series keeps the same row labels as adults.
adults["age_category"] = agegroups


In [ ]:
adults.head()


For each age group, calculate the three mean lengths and check how many
adults contribute. These are group summaries, unlike the individual-level
correlations above.


In [ ]:
# Select only the lengths we intend to average.
summary = adults.groupby("age_category", observed=True)[["height", "leg_length", "arm_length"]].mean()
summary


In [ ]:
# Keep the age labels; do not replace them with assumed numeric positions.
age_counts = adults.groupby("age_category", observed=True).size()
age_counts.rename("n_adults")


In [ ]:
ax = summary.plot(marker="o")
ax.set(title="Mean body lengths by age group", xlabel="Age group (years)", ylabel="Mean length (cm)")
plt.show()


Height is on a different scale from the two limb lengths. To compare how
the group means vary, subtract **128 cm** from mean height before plotting.

This is an arbitrary vertical shift for display. It preserves height differences
between groups; it does not make the three measurements equal or adjust for
other variables.


In [ ]:
# Retain the original height means and add a separate display column.
summary["height_minus_128"] = summary["height"] - 128
ax = summary[["leg_length", "arm_length", "height_minus_128"]].plot(marker="o")
ax.set(
    title="Age-group means with height shifted down by 128 cm",
    xlabel="Age group (years)", ylabel="Mean length (cm; height shifted)",
)
plt.show()


### Comments

Older age groups in this sample have lower mean height and upper-leg length;
mean upper-arm length changes relatively little. The individual correlations
also show that upper-arm length is strongly associated with height, even though
its age-group means are fairly similar.

We are comparing **different people**, not following people as they age. These
cross-sectional differences alone do not show how an individual's body changes
over time; group composition and birth-cohort differences may also matter.


## Is there a relationship between height or weight and income?

In [ ]:
# Use the same adult records for the height and weight comparisons.
income_weight = merged.loc[
    merged["RIDAGEYR"] >= 20, ["BMXHT", "BMXWT", "INDHHIN2", "RIDAGEYR"]
].copy()
print("Adult records before missing-value exclusions:", len(income_weight))
print(income_weight.isna().sum())
adults_inc_wt = income_weight.dropna(subset=["BMXHT", "BMXWT", "INDHHIN2", "RIDAGEYR"]).copy()
print("Adults with all selected fields recorded:", len(adults_inc_wt))
adults_inc_wt.info()


In [ ]:
# A code can be nonmissing but still mean "refused" or "don't know".
adults_inc_wt["INDHHIN2"].value_counts(dropna=False).sort_index()


`INDHHIN2` records **household income brackets**, not an individual's salary.
Codes 77 and 99 mean refused and don't know. Codes 12 and 13 are valid but broad
answers (20,000 dollars or more, and under 20,000 dollars) that overlap the detailed
brackets. We exclude those four codes for this ordered-bracket display.

Map the retained codes to their dollar labels. Bracket widths differ, and the
highest is open-ended, so equal plot spacing represents categories rather than
equal dollar differences. Check the group counts alongside the means.

[Income codebook](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/DEMO_I.htm#INDHHIN2)


In [ ]:
# Dictionary order follows the detailed brackets from lowest to highest.
income_labels = {
    1: "0-4,999", 2: "5,000-9,999", 3: "10,000-14,999",
    4: "15,000-19,999", 5: "20,000-24,999", 6: "25,000-34,999",
    7: "35,000-44,999", 8: "45,000-54,999", 9: "55,000-64,999",
    10: "65,000-74,999", 14: "75,000-99,999", 15: "100,000+",
}
income_order = list(income_labels)
detailed_income = adults_inc_wt.loc[adults_inc_wt["INDHHIN2"].isin(income_order)].copy()
print("Adults in detailed brackets:", len(detailed_income))
print("Other income codes excluded:", len(adults_inc_wt) - len(detailed_income))


In [ ]:
income_summary = detailed_income.groupby("INDHHIN2").agg(
    n_adults=("BMXWT", "size"),
    mean_weight_kg=("BMXWT", "mean"),
    mean_height_cm=("BMXHT", "mean"),
).reindex(income_order)
# Change the display labels without treating code numbers as dollar amounts.
income_summary = income_summary.rename(index=income_labels)
income_summary.index.name = "household_income"
income_summary


In [ ]:
# The x-axis is an ordered set of brackets, not numeric income-code distances.
ax = income_summary["mean_weight_kg"].plot(marker="o", figsize=(10, 4), rot=45)
ax.set(
    title="Mean recorded adult weight by household income bracket",
    xlabel="Annual household income ($; ordered brackets)", ylabel="Mean weight (kg)",
)
# Show every bracket label, including the highest income category.
ax.set_xticks(range(len(income_summary)), labels=income_summary.index, rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
ax = income_summary["mean_height_cm"].plot(marker="o", figsize=(10, 4), rot=45)
ax.set(
    title="Mean recorded adult height by household income bracket",
    xlabel="Annual household income ($; ordered brackets)", ylabel="Mean height (cm)",
)
# Show every bracket label, including the highest income category.
ax.set_xticks(range(len(income_summary)), labels=income_summary.index, rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Comments

Mean height generally rises across the income brackets in this sample; mean
weight has no similarly clear increasing pattern. These are unadjusted sample
comparisons, so they do not isolate an effect of household income. For example,
age and sex composition may differ across income groups.

A next question could be: does the height pattern persist when comparing
similar age and sex groups? The plot helps us choose a follow-up question.
